In [1]:
# ===== Imports and settings =====
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
RISK_ORDER = ["High", "Medium", "Low"]

def assign_risk(g3):
    if g3 <= 9:
        return "High"
    elif g3 <= 13:
        return "Medium"
    return "Low"

In [2]:
# ===== Pooled data with a subject indicator and student groups =====
mat = pd.read_csv("../data/raw/student-mat.csv", sep=";")
por = pd.read_csv("../data/raw/student-por.csv", sep=";")
mat["subject"] = 0
por["subject"] = 1

pooled = pd.concat([mat, por], ignore_index=True)
pooled["risk"] = pooled["G3"].apply(assign_risk)

keys = ["school", "sex", "age", "address", "famsize", "Pstatus",
        "Medu", "Fedu", "Mjob", "Fjob", "reason", "nursery", "internet"]
pooled["student_group"] = pooled.groupby(keys).ngroup()

# Engineered grade features
pooled["grade_change"] = pooled["G2"] - pooled["G1"]
pooled["grade_mean"] = (pooled["G1"] + pooled["G2"]) / 2
pooled["G1_x_subject"] = pooled["G1"] * pooled["subject"]
pooled["G2_x_subject"] = pooled["G2"] * pooled["subject"]

print("Rows:", len(pooled), " Student groups:", pooled["student_group"].nunique())

Rows: 1044  Student groups: 662


In [3]:
# ===== Final test set, split by student group and left untouched =====
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
dev_idx, final_idx = next(gss.split(pooled, pooled["risk"], pooled["student_group"]))

dev = pooled.iloc[dev_idx].reset_index(drop=True)
final_test = pooled.iloc[final_idx].reset_index(drop=True)
final_test.to_csv("../data/processed/final_test_do_not_open.csv", index=False)

print("Development:", len(dev), " Final test:", len(final_test))
print(dev["risk"].value_counts().reindex(RISK_ORDER).to_dict())
print(final_test["risk"].value_counts().reindex(RISK_ORDER).to_dict())

Development: 834  Final test: 210
{'High': 184, 'Medium': 412, 'Low': 238}
{'High': 46, 'Medium': 108, 'Low': 56}


In [4]:
# ===== Feature sets for the two stages =====
context = ["studytime", "failures", "absences", "famrel", "freetime",
           "goout", "Dalc", "Walc", "health", "traveltime"]

stages = {
    "Stage 1, G1 only": (["G1", "subject", "G1_x_subject"], "G1"),
    "Stage 1, G1 and context": (["G1", "subject", "G1_x_subject"] + context, "G1"),
    "Stage 2, G1 and G2": (["G1", "G2", "grade_change", "grade_mean",
                            "subject", "G1_x_subject", "G2_x_subject"], "G2"),
    "Stage 2, G1, G2 and context": (["G1", "G2", "grade_change", "grade_mean",
                                     "subject", "G1_x_subject", "G2_x_subject"] + context, "G2"),
}

models = {
    "Logistic regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                            random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                                                        max_iter=200, random_state=RANDOM_STATE),
}

In [5]:
# ===== Cross validation and scoring helpers =====
def out_of_fold(X, y, groups, model, repeats=3):
    runs = []
    for seed in range(repeats):
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + seed)
        pred = np.empty(len(y), dtype=object)
        for tr, te in cv.split(X, y, groups):
            pipe = Pipeline([("scale", StandardScaler()), ("clf", clone(model))])
            pipe.fit(X.iloc[tr], y.iloc[tr])
            pred[te] = pipe.predict(X.iloc[te])
        runs.append(pred)
    return runs


def score(y, pred, rule, mask):
    yy, pp = y[mask], pred[mask]
    ok_model = (pred == y) & mask
    ok_rule = (rule == y) & mask
    return {
        "recall_high": recall_score(yy, pp, labels=["High"], average="macro", zero_division=0),
        "precision_high": precision_score(yy, pp, labels=["High"], average="macro", zero_division=0),
        "f1_macro": f1_score(yy, pp, average="macro"),
        "accuracy": accuracy_score(yy, pp),
        "high_predicted_low": int(((y == "High") & (pred == "Low") & mask).sum()),
        "net_vs_rule": int((ok_model & ~ok_rule).sum() - (~ok_model & ok_rule).sum()),
    }

In [6]:
# ===== Run all stages, models and the rule baseline =====
y = dev["risk"]
groups = dev["student_group"]

subject_masks = {
    "All": np.ones(len(dev), dtype=bool),
    "Mathematics": (dev["subject"] == 0).values,
    "Portuguese": (dev["subject"] == 1).values,
}

y_arr = y.values
records = []

for stage, (cols, rule_col) in stages.items():
    X = dev[cols]
    rule_pred = dev[rule_col].apply(assign_risk).values

    for subj, mask in subject_masks.items():
        records.append({"stage": stage, "approach": "Rule, carry grade forward",
                        "subject": subj, **score(y_arr, rule_pred, rule_pred, mask)})

    for name, model in models.items():
        runs = out_of_fold(X, y, groups, model)
        for subj, mask in subject_masks.items():
            per_run = pd.DataFrame([score(y_arr, pred, rule_pred, mask) for pred in runs])
            records.append({"stage": stage, "approach": name,
                            "subject": subj, **per_run.mean().to_dict()})

results_pooled = pd.DataFrame(records).round(3)
results_pooled.to_csv("../outputs/tables/pooled_experiments.csv", index=False)
results_pooled[results_pooled["subject"] == "All"]

,stage,approach,subject,recall_high,precision_high,f1_macro,accuracy,high_predicted_low,net_vs_rule
0,"Stage 1, G1 only","Rule, carry grade forward",All,0.826,0.644,0.742,0.740,0.000,0.000
3,"Stage 1, G1 only",Logistic regression,All,0.777,0.673,0.730,0.727,0.000,-11.000
6,"Stage 1, G1 only",Random Forest,All,0.864,0.639,0.736,0.730,0.000,-8.333
9,"Stage 1, G1 only",Gradient boosting,All,0.719,0.707,0.743,0.748,0.000,7.000
12,"Stage 1, G1 and context","Rule, carry grade forward",All,0.826,0.644,0.742,0.740,0.000,0.000
15,"Stage 1, G1 and context",Logistic regression,All,0.801,0.658,0.734,0.729,0.000,-8.667
18,"Stage 1, G1 and context",Random Forest,All,0.777,0.682,0.740,0.738,1.000,-1.667
21,"Stage 1, G1 and context",Gradient boosting,All,0.694,0.724,0.737,0.741,0.333,1.333
24,"Stage 2, G1 and G2","Rule, carry grade forward",All,0.918,0.741,0.849,0.849,0.000,0.000
27,"Stage 2, G1 and G2",Logistic regression,All,0.895,0.779,0.841,0.839,0.000,-8.333


In [7]:
# ===== Look at the subjects separately =====
results_pooled[results_pooled["subject"] == "Portuguese"]

,stage,approach,subject,recall_high,precision_high,f1_macro,accuracy,high_predicted_low,net_vs_rule
2,"Stage 1, G1 only","Rule, carry grade forward",Portuguese,0.862,0.575,0.742,0.753,0.0,0.000
5,"Stage 1, G1 only",Logistic regression,Portuguese,0.729,0.648,0.744,0.757,0.0,2.333
8,"Stage 1, G1 only",Random Forest,Portuguese,0.737,0.618,0.736,0.749,0.0,-2.000
11,"Stage 1, G1 only",Gradient boosting,Portuguese,0.650,0.788,0.767,0.792,0.0,20.000
14,"Stage 1, G1 and context","Rule, carry grade forward",Portuguese,0.862,0.575,0.742,0.753,0.0,0.000
17,"Stage 1, G1 and context",Logistic regression,Portuguese,0.708,0.612,0.719,0.732,0.0,-10.667
20,"Stage 1, G1 and context",Random Forest,Portuguese,0.696,0.664,0.730,0.745,0.0,-4.333
23,"Stage 1, G1 and context",Gradient boosting,Portuguese,0.608,0.668,0.714,0.740,0.0,-6.667
26,"Stage 2, G1 and G2","Rule, carry grade forward",Portuguese,0.888,0.634,0.814,0.828,0.0,0.000
29,"Stage 2, G1 and G2",Logistic regression,Portuguese,0.833,0.739,0.822,0.830,0.0,1.000


In [8]:
# ===== Threshold free comparison for the High class =====
from sklearn.metrics import average_precision_score, precision_recall_curve

def oof_proba_high(X, y, groups, model, seed=0):
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + seed)
    scores = np.zeros(len(y))
    for tr, te in cv.split(X, y, groups):
        pipe = Pipeline([("scale", StandardScaler()), ("clf", clone(model))])
        pipe.fit(X.iloc[tr], y.iloc[tr])
        proba = pipe.predict_proba(X.iloc[te])
        scores[te] = proba[:, list(pipe.classes_).index("High")]
    return scores


def precision_at_recall(is_high, score, target=0.90):
    prec, rec, _ = precision_recall_curve(is_high, score)
    return prec[rec >= target].max()


is_high = (dev["risk"] == "High").astype(int).values
rows = []

for stage in ["Stage 1, G1 only", "Stage 1, G1 and context", "Stage 2, G1 and G2"]:
    cols, rule_col = stages[stage]
    X = dev[cols]

    candidates = {"Rule score (lower grade is riskier)": [-dev[rule_col].values]}
    for name, model in models.items():
        candidates[name] = [oof_proba_high(X, y, groups, model, seed=s) for s in range(3)]

    for name, score_list in candidates.items():
        for subj, mask in subject_masks.items():
            rows.append({
                "stage": stage,
                "approach": name,
                "subject": subj,
                "avg_precision": np.mean([average_precision_score(is_high[mask], s[mask])
                                          for s in score_list]),
                "precision_at_90_recall": np.mean([precision_at_recall(is_high[mask], s[mask])
                                                   for s in score_list]),
            })

results_curve = pd.DataFrame(rows).round(3)
results_curve.to_csv("../outputs/tables/pooled_threshold_free.csv", index=False)
results_curve

,stage,approach,subject,avg_precision,precision_at_90_recall
0,"Stage 1, G1 only",Rule score (lower grade is riskier),All,0.752,0.510
1,"Stage 1, G1 only",Rule score (lower grade is riskier),Mathematics,0.813,0.654
2,"Stage 1, G1 only",Rule score (lower grade is riskier),Portuguese,0.707,0.398
3,"Stage 1, G1 only",Logistic regression,All,0.777,0.622
4,"Stage 1, G1 only",Logistic regression,Mathematics,0.824,0.666
5,"Stage 1, G1 only",Logistic regression,Portuguese,0.716,0.462
6,"Stage 1, G1 only",Random Forest,All,0.770,0.619
7,"Stage 1, G1 only",Random Forest,Mathematics,0.799,0.655
8,"Stage 1, G1 only",Random Forest,Portuguese,0.689,0.456
9,"Stage 1, G1 only",Gradient boosting,All,0.768,0.619


In [9]:
# ===== Stronger simple baselines =====
def lr_scores(cols, seeds=range(3)):
    return [oof_proba_high(dev[cols], y, groups, models["Logistic regression"], seed=s)
            for s in seeds]

stage2_cols = stages["Stage 2, G1 and G2"][0]

candidates = {
    "Rule, G2 only": [-dev["G2"].values],
    "Average of G1 and G2": [-(dev["G1"] + dev["G2"]).values / 2],
    "Logistic, G2 and subject": lr_scores(["G2", "subject", "G2_x_subject"]),
    "Logistic, full Stage 2": lr_scores(stage2_cols),
}

rows = []
for name, score_list in candidates.items():
    for subj, mask in subject_masks.items():
        rows.append({
            "approach": name,
            "subject": subj,
            "avg_precision": np.mean([average_precision_score(is_high[mask], s[mask])
                                      for s in score_list]),
            "precision_at_90_recall": np.mean([precision_at_recall(is_high[mask], s[mask])
                                               for s in score_list]),
        })

results_baselines = pd.DataFrame(rows).round(3)
results_baselines.to_csv("../outputs/tables/pooled_stronger_baselines.csv", index=False)
results_baselines

,approach,subject,avg_precision,precision_at_90_recall
0,"Rule, G2 only",All,0.877,0.741
1,"Rule, G2 only",Mathematics,0.933,0.845
2,"Rule, G2 only",Portuguese,0.799,0.436
3,Average of G1 and G2,All,0.879,0.672
4,Average of G1 and G2,Mathematics,0.918,0.762
5,Average of G1 and G2,Portuguese,0.834,0.583
6,"Logistic, G2 and subject",All,0.909,0.741
7,"Logistic, G2 and subject",Mathematics,0.943,0.845
8,"Logistic, G2 and subject",Portuguese,0.818,0.564
9,"Logistic, full Stage 2",All,0.920,0.769


In [10]:
# ===== Bootstrap gap in average precision, logistic against the plain average =====
rng = np.random.default_rng(RANDOM_STATE)

def bootstrap_gap(score_a, score_b, mask, n_boot=500):
    idx = np.where(mask)[0]
    gaps = []
    for _ in range(n_boot):
        sample = rng.choice(idx, size=len(idx), replace=True)
        if is_high[sample].sum() < 5:
            continue
        gaps.append(average_precision_score(is_high[sample], score_a[sample])
                    - average_precision_score(is_high[sample], score_b[sample]))
    return np.percentile(gaps, [2.5, 50, 97.5]).round(3)

full_lr = candidates["Logistic, full Stage 2"][0]
average = candidates["Average of G1 and G2"][0]

for subj in ["Mathematics", "Portuguese"]:
    low, mid, high = bootstrap_gap(full_lr, average, subject_masks[subj])
    print(f"{subj}: median gap {mid}, 95 percent interval {low} to {high}")

Mathematics: median gap 0.031, 95 percent interval 0.016 to 0.052
Portuguese: median gap 0.039, 95 percent interval 0.011 to 0.072


In [11]:
# ===== Bootstrap against the strongest simple baseline in each subject =====
rule_scores = candidates["Rule, G2 only"][0]
best_simple = {"Mathematics": rule_scores, "Portuguese": average}

for subj in ["Mathematics", "Portuguese"]:
    low, mid, high = bootstrap_gap(full_lr, best_simple[subj], subject_masks[subj])
    print(f"{subj}: median gap {mid}, 95 percent interval {low} to {high}")

Mathematics: median gap 0.016, 95 percent interval 0.008 to 0.027
Portuguese: median gap 0.037, 95 percent interval 0.008 to 0.073


In [12]:
# ===== Do the engineered columns matter? =====
core_cols = ["G1", "G2", "subject", "G1_x_subject", "G2_x_subject"]
candidates["Logistic, core columns"] = lr_scores(core_cols)

rows = []
for name in ["Logistic, core columns", "Logistic, full Stage 2"]:
    for subj, mask in subject_masks.items():
        rows.append({
            "approach": name,
            "subject": subj,
            "avg_precision": np.mean([average_precision_score(is_high[mask], s[mask])
                                      for s in candidates[name]]),
            "precision_at_90_recall": np.mean([precision_at_recall(is_high[mask], s[mask])
                                               for s in candidates[name]]),
        })

pd.DataFrame(rows).round(3)

,approach,subject,avg_precision,precision_at_90_recall
0,"Logistic, core columns",All,0.916,0.746
1,"Logistic, core columns",Mathematics,0.948,0.835
2,"Logistic, core columns",Portuguese,0.864,0.687
3,"Logistic, full Stage 2",All,0.920,0.769
4,"Logistic, full Stage 2",Mathematics,0.949,0.853
5,"Logistic, full Stage 2",Portuguese,0.872,0.660


In [13]:
# ===== What did the model learn? Coefficients in grade points =====
lr = LogisticRegression(max_iter=2000, class_weight="balanced")
lr.fit(dev[core_cols], dev["risk"])

high_idx = list(lr.classes_).index("High")
coef = pd.Series(lr.coef_[high_idx], index=core_cols).round(3)
print(coef)

print("\nEffective weight on each grade for High risk")
print("Mathematics:  G1", round(coef["G1"], 3), " G2", round(coef["G2"], 3))
print("Portuguese:   G1", round(coef["G1"] + coef["G1_x_subject"], 3),
      " G2", round(coef["G2"] + coef["G2_x_subject"], 3))

G1             -0.130
G2             -1.858
subject         0.081
G1_x_subject   -0.247
G2_x_subject    0.167
dtype: float64

Effective weight on each grade for High risk
Mathematics:  G1 -0.13  G2 -1.858
Portuguese:   G1 -0.377  G2 -1.691


In [14]:
# ===== Add the proposed Decision Tree and KNN to the pooled comparison =====
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

models_extended = {
    **models,
    "Decision Tree": DecisionTreeClassifier(max_depth=3, min_samples_leaf=10,
                                            class_weight="balanced",
                                            random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=15),
}

compare_stages = ["Stage 1, G1 only", "Stage 2, G1 and G2"]
rows = []

for stage in compare_stages:
    cols, rule_col = stages[stage]
    X = dev[cols]

    for name, model in models_extended.items():
        score_list = [oof_proba_high(X, y, groups, model, seed=s) for s in range(3)]

        for subj, mask in subject_masks.items():
            rows.append({
                "stage": stage,
                "model": name,
                "subject": subj,
                "avg_precision": np.mean([average_precision_score(is_high[mask], s[mask])
                                          for s in score_list]),
                "precision_at_90_recall": np.mean([precision_at_recall(is_high[mask], s[mask])
                                                   for s in score_list]),
            })

results_all_models = pd.DataFrame(rows).round(3)
results_all_models.to_csv("../outputs/tables/pooled_all_models.csv", index=False)

summary = (results_all_models[results_all_models["subject"] != "All"]
           .pivot_table(index=["stage", "model"], columns="subject",
                        values="avg_precision"))
summary["mean_of_subjects"] = summary.mean(axis=1)
summary.round(3).sort_values(["stage", "mean_of_subjects"], ascending=[True, False])

subject                                 Mathematics  Portuguese  \
stage              model                                          
Stage 1, G1 only   Logistic regression        0.824       0.716   
                   KNN                        0.794       0.704   
                   Random Forest              0.799       0.689   
                   Gradient boosting          0.796       0.679   
                   Decision Tree              0.761       0.711   
Stage 2, G1 and G2 Logistic regression        0.949       0.872   
                   Gradient boosting          0.947       0.855   
                   Random Forest              0.940       0.842   
                   KNN                        0.929       0.833   
                   Decision Tree              0.908       0.792   

subject                                 mean_of_subjects  
stage              model                                  
Stage 1, G1 only   Logistic regression             0.770  
                   KNN                             0.749  
                   Random Forest                   0.744  
                   Gradient boosting               0.738  
                   Decision Tree                   0.736  
Stage 2, G1 and G2 Logistic regression             0.910  
                   Gradient boosting               0.901  
                   Random Forest                   0.891  
                   KNN                             0.881  
                   Decision Tree                   0.850

In [15]:
# ===== Bootstrap the grade weights, resampling whole student groups =====
rng = np.random.default_rng(RANDOM_STATE)
group_ids = dev["student_group"].unique()
rows_by_group = dev.groupby("student_group").indices

weights = []
for _ in range(300):
    sampled = rng.choice(group_ids, size=len(group_ids), replace=True)
    idx = np.concatenate([rows_by_group[g] for g in sampled])
    boot = dev.iloc[idx]
    if boot["risk"].nunique() < 3:
        continue

    m = LogisticRegression(max_iter=2000, class_weight="balanced")
    m.fit(boot[core_cols], boot["risk"])
    c = pd.Series(m.coef_[list(m.classes_).index("High")], index=core_cols)

    weights.append({
        "G1 Mathematics": c["G1"],
        "G2 Mathematics": c["G2"],
        "G1 Portuguese": c["G1"] + c["G1_x_subject"],
        "G2 Portuguese": c["G2"] + c["G2_x_subject"],
        "G1 gap, Portuguese minus Mathematics": c["G1_x_subject"],
    })

pd.DataFrame(weights).quantile([0.025, 0.5, 0.975]).T.round(3)

,0.025,0.500,0.975
G1 Mathematics,-0.307,-0.124,0.035
G2 Mathematics,-2.270,-1.904,-1.609
G1 Portuguese,-0.702,-0.392,-0.204
G2 Portuguese,-2.067,-1.703,-1.378
"G1 gap, Portuguese minus Mathematics",-0.593,-0.280,0.004


In [16]:
# ===== Fair tuning, Stage 2 core columns, same grouped folds for every model =====
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

core_cols = ["G1", "G2", "subject", "G1_x_subject", "G2_x_subject"]

def high_avg_precision(estimator, X, y_true):
    proba = estimator.predict_proba(X)
    col = list(estimator.classes_).index("High")
    return average_precision_score((y_true == "High").astype(int), proba[:, col])

tuning_plan = {
    "Logistic regression": (
        LogisticRegression(max_iter=2000, class_weight="balanced"),
        {"clf__C": [0.1, 0.3, 1, 3, 10]},
    ),
    "Decision Tree": (
        DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        {"clf__max_depth": [2, 3, 4, 5], "clf__min_samples_leaf": [5, 10, 20]},
    ),
    "Random Forest": (
        RandomForestClassifier(n_estimators=300, class_weight="balanced",
                               random_state=RANDOM_STATE, n_jobs=-1),
        {"clf__max_depth": [3, 5, 8, None], "clf__min_samples_leaf": [3, 5, 10]},
    ),
    "KNN": (
        KNeighborsClassifier(),
        {"clf__n_neighbors": [9, 15, 25, 35], "clf__weights": ["uniform", "distance"]},
    ),
}

cv_tune = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
tuned = {}

for name, (model, grid) in tuning_plan.items():
    pipe = Pipeline([("scale", StandardScaler()), ("clf", model)])
    search = GridSearchCV(pipe, grid, scoring=high_avg_precision, cv=cv_tune, n_jobs=-1)
    search.fit(dev[core_cols], dev["risk"], groups=dev["student_group"])

    tuned[name] = search.best_estimator_
    rows.append({
        "model": name,
        "avg_precision_high": round(search.best_score_, 3),
        "best_params": {k.replace("clf__", ""): v for k, v in search.best_params_.items()},
    })

pd.DataFrame(rows).sort_values("avg_precision_high", ascending=False)

,model,avg_precision_high,best_params
0,Logistic regression,0.920,{'C': 10}
2,Random Forest,0.920,"{'max_depth': 8, 'min_samples_leaf': 3}"
3,KNN,0.903,"{'n_neighbors': 25, 'weights': 'uniform'}"
1,Decision Tree,0.891,"{'max_depth': 5, 'min_samples_leaf': 10}"


In [17]:
# ===== Per subject check of the tuned models, fresh folds =====
from sklearn.model_selection import cross_val_predict

cv_check = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=7)
rows = []

for name in ["Random Forest", "Logistic regression", "KNN", "Decision Tree"]:
    est = clone(tuned[name])
    proba = cross_val_predict(est, dev[core_cols], dev["risk"], cv=cv_check,
                              groups=dev["student_group"], method="predict_proba")
    est.fit(dev[core_cols], dev["risk"])
    p_high = proba[:, list(est.classes_).index("High")]

    for subj, mask in subject_masks.items():
        rows.append({
            "model": name,
            "subject": subj,
            "avg_precision": average_precision_score(is_high[mask], p_high[mask]),
            "precision_at_90_recall": precision_at_recall(is_high[mask], p_high[mask]),
        })

check = pd.DataFrame(rows).round(3)
check.pivot_table(index="model", columns="subject",
                  values=["avg_precision", "precision_at_90_recall"])

avg_precision                         \
subject                       All Mathematics Portuguese   
model                                                      
Decision Tree               0.889       0.917      0.843   
KNN                         0.899       0.935      0.848   
Logistic regression         0.919       0.944      0.874   
Random Forest               0.904       0.939      0.844   

                    precision_at_90_recall                         
subject                                All Mathematics Portuguese  
model                                                              
Decision Tree                        0.671       0.843      0.576  
KNN                                  0.727       0.847      0.612  
Logistic regression                  0.780       0.855      0.682  
Random Forest                        0.761       0.841      0.643

In [18]:
# ===== Settings to freeze =====
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import cross_val_predict
import joblib

TARGET_RECALL = 0.90

STAGE_COLS = {
    "stage1": ["G1", "subject", "G1_x_subject"],
    "stage2": ["G1", "G2", "subject", "G1_x_subject", "G2_x_subject"],
}

def pick_threshold(is_high_arr, p_high, target):
    p_high = np.round(p_high, 4)
    precision, recall, thresholds = precision_recall_curve(is_high_arr, p_high)
    reachable = np.where(recall[:-1] >= target)[0]
    return float(thresholds[reachable].max())

In [19]:
# ===== Build a frozen bundle for any model template =====
def build_bundle(template, name, target=TARGET_RECALL):
    bundle = {"name": name, "cols": STAGE_COLS, "models": {}, "thresholds": {}}
    is_high_dev = (dev["risk"] == "High").astype(int).values

    for stage, cols in STAGE_COLS.items():
        est = clone(template)
        cv_fit = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=11)
        proba = cross_val_predict(est, dev[cols], dev["risk"], cv=cv_fit,
                                  groups=dev["student_group"], method="predict_proba")

        est.fit(dev[cols], dev["risk"])
        p_high = np.round(proba[:, list(est.classes_).index("High")], 4)
        threshold = pick_threshold(is_high_dev, p_high, target)

        flagged = p_high >= threshold
        print(f"{name}, {stage}: threshold {threshold:.3f}, "
              f"share flagged {flagged.mean():.3f}, "
              f"recall {is_high_dev[flagged].sum() / is_high_dev.sum():.3f}, "
              f"precision {is_high_dev[flagged].mean():.3f}")

        bundle["models"][stage] = est
        bundle["thresholds"][stage] = threshold

    return bundle

bundle_rf = build_bundle(tuned["Random Forest"], "Random Forest")
bundle_lr = build_bundle(tuned["Logistic regression"], "Logistic regression")

joblib.dump(bundle_rf, "../models/spark_bundle_rf.joblib")
joblib.dump(bundle_lr, "../models/spark_bundle_lr.joblib")

Random Forest, stage1: threshold 0.485, share flagged 0.327, recall 0.918, precision 0.619
Random Forest, stage2: threshold 0.504, share flagged 0.253, recall 0.902, precision 0.787
Logistic regression, stage1: threshold 0.458, share flagged 0.327, recall 0.918, precision 0.619
Logistic regression, stage2: threshold 0.453, share flagged 0.253, recall 0.902, precision 0.787


['../models/spark_bundle_lr.joblib']

In [20]:
# ===== Scoring function for new student data =====
SUBJECTS = {"mat": 0, "math": 0, "mathematics": 0, "0": 0,
            "por": 1, "portuguese": 1, "1": 1}

def score_students(new_data, bundle):
    data = new_data.copy()

    missing = [c for c in ["subject", "G1"] if c not in data.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    if "G2" not in data.columns:
        data["G2"] = np.nan

    subject = data["subject"].astype(str).str.strip().str.lower().map(SUBJECTS)
    if subject.isna().any():
        raise ValueError("subject must be Mathematics or Portuguese")
    data["subject"] = subject

    for grade in ["G1", "G2"]:
        data[grade] = pd.to_numeric(data[grade], errors="coerce")
    if data["G1"].isna().any():
        raise ValueError("G1 is required for every student")

    out_of_range = ((data[["G1", "G2"]] < 0) | (data[["G1", "G2"]] > 20)).any(axis=1)
    if out_of_range.any():
        raise ValueError("Grades must be between 0 and 20")

    data["G1_x_subject"] = data["G1"] * data["subject"]
    data["G2_x_subject"] = data["G2"] * data["subject"]

    result = pd.DataFrame(index=data.index, columns=["stage", "p_high", "risk"], dtype=object)
    has_g2 = data["G2"].notna()

    for stage, mask in [("stage2", has_g2), ("stage1", ~has_g2)]:
        if not mask.any():
            continue

        model = bundle["models"][stage]
        classes = list(model.classes_)
        proba = model.predict_proba(data.loc[mask, bundle["cols"][stage]])

        p_high = np.round(proba[:, classes.index("High")], 4)
        p_medium = proba[:, classes.index("Medium")]
        p_low = proba[:, classes.index("Low")]

        risk = np.where(p_high >= bundle["thresholds"][stage], "High",
                        np.where(p_medium >= p_low, "Medium", "Low"))

        result.loc[mask, "stage"] = stage
        result.loc[mask, "p_high"] = p_high.round(3)
        result.loc[mask, "risk"] = risk

    return result

In [21]:
# ===== Quick check on invented students =====
examples = pd.DataFrame({
    "subject": ["Mathematics", "Portuguese", "Portuguese", "Mathematics"],
    "G1": [8, 11, 14, 12],
    "G2": [7, 10, np.nan, 12],
})
score_students(examples, bundle_rf)

,stage,p_high,risk
0,stage2,0.984,High
1,stage2,0.003,Medium
2,stage1,0.0,Low
3,stage2,0.008,Medium


In [22]:
# ===== Do the two bundles flag the same students? =====
def oof_flags(bundle, stage):
    cols = STAGE_COLS[stage]
    classes = sorted(dev["risk"].unique())
    cv_fit = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=11)
    proba = cross_val_predict(clone(bundle["models"][stage]), dev[cols], dev["risk"],
                              cv=cv_fit, groups=dev["student_group"],
                              method="predict_proba")
    return np.round(proba[:, classes.index("High")], 4) >= bundle["thresholds"][stage]

for stage, col in [("stage1", "G1"), ("stage2", "G2")]:
    rf_flag = oof_flags(bundle_rf, stage)
    lr_flag = oof_flags(bundle_lr, stage)

    print(stage)
    print("  flagged by both:", int((rf_flag & lr_flag).sum()),
          " forest only:", int((rf_flag & ~lr_flag).sum()),
          " logistic only:", int((~rf_flag & lr_flag).sum()))

    share = (dev.assign(flag=rf_flag)
                .groupby(["subject", col])["flag"].mean()
                .unstack(0).round(2))
    share.columns = ["Mathematics", "Portuguese"]
    print(share[(share > 0).any(axis=1)])

stage1
  flagged by both: 273  forest only: 0  logistic only: 0
    Mathematics  Portuguese
G1                         
0           NaN         1.0
3           1.0         NaN
4           1.0         1.0
5           1.0         1.0
6           1.0         1.0
7           1.0         1.0
8           1.0         1.0
9           1.0         1.0
10          1.0         0.0
stage2
  flagged by both: 207  forest only: 4  logistic only: 4
    Mathematics  Portuguese
G2                         
0           1.0        1.00
4           1.0         NaN
5           1.0        1.00
6           1.0        1.00
7           1.0        1.00
8           1.0        1.00
9           1.0        0.59
10          0.0        0.07


In [23]:
# ===== Are many students sitting exactly on the threshold? =====
for stage in ["stage1", "stage2"]:
    cols = STAGE_COLS[stage]
    classes = sorted(dev["risk"].unique())
    cv_fit = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=11)

    proba = cross_val_predict(clone(bundle_rf["models"][stage]), dev[cols], dev["risk"],
                              cv=cv_fit, groups=dev["student_group"],
                              method="predict_proba")
    p_high = proba[:, classes.index("High")]
    threshold = bundle_rf["thresholds"][stage]

    print(stage,
          " development students:", len(dev),
          " flagged now:", int((p_high >= threshold).sum()),
          " within 1e-6 of threshold:", int((np.abs(p_high - threshold) < 1e-6).sum()))

stage1  development students: 834  flagged now: 260  within 1e-6 of threshold: 0
stage2  development students: 834  flagged now: 210  within 1e-6 of threshold: 0


In [24]:
# ===== Threshold check using the same rounding as the pipeline =====
for stage in ["stage1", "stage2"]:
    cols = STAGE_COLS[stage]
    classes = sorted(dev["risk"].unique())
    cv_fit = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=11)

    proba = cross_val_predict(clone(bundle_rf["models"][stage]), dev[cols], dev["risk"],
                              cv=cv_fit, groups=dev["student_group"],
                              method="predict_proba")
    p_high = np.round(proba[:, classes.index("High")], 4)
    threshold = bundle_rf["thresholds"][stage]

    print(stage,
          " flagged:", int((p_high >= threshold).sum()),
          " share:", round((p_high >= threshold).mean(), 3),
          " exactly on threshold:", int((p_high == threshold).sum()))

stage1  flagged: 273  share: 0.327  exactly on threshold: 13
stage2  flagged: 211  share: 0.253  exactly on threshold: 1


In [25]:
# ===== Final test, run once after everything is frozen =====
final = pd.read_csv("../data/processed/final_test_do_not_open.csv")
final_high = (final["risk"] == "High").astype(int).values
final_subject = final["subject"].values

def final_report(pred, label):
    rows = []
    for subj_name, mask in [("All", np.ones(len(final), dtype=bool)),
                            ("Mathematics", final_subject == 0),
                            ("Portuguese", final_subject == 1)]:
        yy, pp = final["risk"].values[mask], np.asarray(pred)[mask]
        rows.append({
            "approach": label, "subject": subj_name, "n": int(mask.sum()),
            "high_students": int((yy == "High").sum()),
            "recall_high": recall_score(yy, pp, labels=["High"], average="macro", zero_division=0),
            "precision_high": precision_score(yy, pp, labels=["High"], average="macro", zero_division=0),
            "f1_macro": f1_score(yy, pp, average="macro"),
            "accuracy": accuracy_score(yy, pp),
        })
    return pd.DataFrame(rows)

frames = [
    final_report(final["G2"].apply(assign_risk).values, "Rule, G2 carried forward"),
    final_report(score_students(final[["subject", "G1", "G2"]], bundle_rf)["risk"].values,
                 "Random Forest, Stage 2"),
    final_report(score_students(final[["subject", "G1", "G2"]], bundle_lr)["risk"].values,
                 "Logistic regression, Stage 2"),
    final_report(score_students(final[["subject", "G1"]], bundle_rf)["risk"].values,
                 "Random Forest, Stage 1"),
]

final_results = pd.concat(frames).round(3)
final_results.to_csv("../outputs/tables/final_test_results.csv", index=False)
final_results

,approach,subject,n,high_students,recall_high,precision_high,f1_macro,accuracy
0,"Rule, G2 carried forward",All,210,46,0.913,0.667,0.825,0.824
1,"Rule, G2 carried forward",Mathematics,79,26,0.923,0.800,0.869,0.861
2,"Rule, G2 carried forward",Portuguese,131,20,0.900,0.545,0.786,0.802
0,"Random Forest, Stage 2",All,210,46,0.913,0.724,0.829,0.829
1,"Random Forest, Stage 2",Mathematics,79,26,0.923,0.800,0.869,0.861
2,"Random Forest, Stage 2",Portuguese,131,20,0.900,0.643,0.797,0.809
0,"Logistic regression, Stage 2",All,210,46,0.913,0.750,0.808,0.805
1,"Logistic regression, Stage 2",Mathematics,79,26,0.923,0.800,0.855,0.848
2,"Logistic regression, Stage 2",Portuguese,131,20,0.900,0.692,0.778,0.779
0,"Random Forest, Stage 1",All,210,46,0.957,0.571,0.709,0.700


In [26]:
# ===== Extra detail on the final test, no settings changed =====
from sklearn.metrics import confusion_matrix

actual = final["risk"].values

predictions_final = {
    "Rule, G2 carried forward": final["G2"].apply(assign_risk).values,
    "Random Forest, Stage 2": score_students(final[["subject", "G1", "G2"]], bundle_rf)["risk"].values,
    "Random Forest, Stage 1": score_students(final[["subject", "G1"]], bundle_rf)["risk"].values,
}

for label, pred in predictions_final.items():
    matrix = pd.DataFrame(
        confusion_matrix(actual, pred, labels=RISK_ORDER),
        index=["true " + r for r in RISK_ORDER],
        columns=["pred " + r for r in RISK_ORDER],
    )
    print(label)
    print(matrix)
    print("High predicted as Low:", int(((actual == "High") & (pred == "Low")).sum()),
          " Flagged High:", int((pred == "High").sum()),
          " False alarms:", int(((pred == "High") & (actual != "High")).sum()))
    print()

Rule, G2 carried forward
             pred High  pred Medium  pred Low
true High           42            4         0
true Medium         21           87         0
true Low             0           12        44
High predicted as Low: 0  Flagged High: 63  False alarms: 21

Random Forest, Stage 2
             pred High  pred Medium  pred Low
true High           42            4         0
true Medium         16           86         6
true Low             0           10        46
High predicted as Low: 0  Flagged High: 58  False alarms: 16

Random Forest, Stage 1
             pred High  pred Medium  pred Low
true High           44            2         0
true Medium         33           58        17
true Low             0           11        45
High predicted as Low: 0  Flagged High: 77  False alarms: 33



In [30]:
# ===== Interactive test of the frozen Random Forest bundle =====
def ask_subject():
    while True:
        text = input("Subject (mat or por, q to quit): ").strip().lower()
        if text == "q":
            return None
        if text in SUBJECTS:
            return text
        print("  Please type mat or por.")


def ask_grade(prompt, required):
    while True:
        text = input(prompt).strip()
        if text == "" and not required:
            return np.nan
        try:
            value = float(text)
        except ValueError:
            print("  Please enter a number between 0 and 20.")
            continue
        if 0 <= value <= 20:
            return value
        print("  Grade must be between 0 and 20.")


def explain_prediction(bundle, subject, g1, g2):
    row = pd.DataFrame({"subject": [subject], "G1": [g1], "G2": [g2]})
    result = score_students(row, bundle).iloc[0]
    stage = result["stage"]

    # Rebuild the model inputs to show every class probability
    code = SUBJECTS[subject]
    features = pd.DataFrame({
        "G1": [g1],
        "G2": [g2],
        "subject": [code],
        "G1_x_subject": [g1 * code],
        "G2_x_subject": [g2 * code],
    })

    model = bundle["models"][stage]
    proba = model.predict_proba(features[bundle["cols"][stage]])[0]
    by_class = dict(zip(model.classes_, np.round(proba, 3)))
    threshold = bundle["thresholds"][stage]

    print("\n----- Result -----")
    print("Stage used        :", "Stage 2 (G1 and G2)" if stage == "stage2" else "Stage 1 (G1 only)")
    print("Predicted risk    :", result["risk"])
    print("Probabilities     : High", by_class["High"],
          "| Medium", by_class["Medium"], "| Low", by_class["Low"])
    print("High threshold    :", round(threshold, 4))

    distance = abs(by_class["High"] - threshold)
    if distance < 0.10:
        print("Note              : close to the High threshold, treat as borderline")
    print("------------------\n")


while True:
    subject = ask_subject()
    if subject is None:
        print("Finished.")
        break

    g1 = ask_grade("G1 (0 to 20): ", required=True)
    g2 = ask_grade("G2 (0 to 20, press Enter to skip): ", required=False)

    explain_prediction(bundle_lr, subject, g1, g2)


----- Result -----
Stage used        : Stage 1 (G1 only)
Predicted risk    : Low
Probabilities     : High 0.036 | Medium 0.472 | Low 0.492
High threshold    : 0.4578
------------------


----- Result -----
Stage used        : Stage 1 (G1 only)
Predicted risk    : High
Probabilities     : High 0.997 | Medium 0.003 | Low 0.0
High threshold    : 0.4578
------------------

Finished.
